In [ ]:
# Parameters
configfile = "config/config.yml"
input_data = "results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_crossmatch.pq"
targetpop_data = "results/data/checkpoints/targetpop.pq"
display_util = "workflow/scripts/display_util.py"
util = "workflow/scripts/util.py"
output_data = "results/data/intermediate_spender_postmortem_labor_crossmatch.pq"
output_model = "results/data/intermediate_spender_postmortem_labor_crossmatch.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    EmpfaengerID,
    drop_duplicate_columns,
    common_translate,
)

### Target Population Filtering

The donor and recipient pairs in the dataset were filtered to match the target population (see [](general:tpf)). We renamed `donor_et_dso` to `donor_et_id_et`. Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
orpairs = data.loc[:, ["recipient_et_id_et", "donor_et_dso"]].drop_duplicates().shape[0]
targetpop = pd.read_parquet(targetpop_data)
poppairs = (
    targetpop.loc[:, ["recipient_et_id_et", "donor_et_id_et"]]
    .drop_duplicates()
    .shape[0]
)
data = pd.merge(
    targetpop.loc[:, ["recipient_et_id_et", "donor_et_id_et"]],
    data,
    how="inner",
    left_on=["recipient_et_id_et", "donor_et_id_et"],
    right_on=["recipient_et_id_et", "donor_et_dso"],
).drop(columns="donor_et_dso")
curpairs = (
    data.loc[:, ["recipient_et_id_et", "donor_et_id_et"]].drop_duplicates().shape[0]
)
display(
    Markdown(
        f"""The filter process reduced the number of pairs in the data ({orpairs}) and target population ({poppairs})
            to {curpairs} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

Only the {term} `DSO` contributed to this dataset (see [](general:ic)). 

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

Some results were removed (see [](general:rf)).

In [ ]:
before = data.shape[0]
filtermask = data.loc[:, "crossmatch_program"].str.contains("Niere")
data = data.loc[filtermask].copy()
after = data.shape[0]
display(
    Markdown(
        f"""We removed results not related to kidney crossmatch from {before} to {after}.
        """
    )
)

Sometimes, there are results from the donor and the recipient region. Regions have their own labs.

In [ ]:
display_long_data_doc(
    data, ["recipient_et_id_et", "donor_et_id_et"], "result_date", "crossmatch_program"
)

### Unit Conversions

We applied common translations and also replaced the status results (see [](general:uc)).

In [ ]:
unknown = ["Serum fehlt", "Nicht auswertbar", "nicht auswertbar"]
additional = [x + "=unknown" for x in unknown]
not_tested = ["bereits getestet"]
additional += [x + "=not tested" for x in not_tested]
additional += ["Negativ=negative", "Positiv=positive"]

data = common_translate(data, config["data"]["common_translations"] + additional)

### Consolidating Columns

No consolidation was necessary, as the dataset only contained {term}`DSO` data (see [](general:crc)).

## Intermediate Dataset

For this longitudinal dataset we recommend the `result_date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et", "recipient_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["result_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important

testres = ["negative", "positive", "unknown", "not tested"]


class DonorPostmortemLabCrossmatch(EmpfaengerID, SpenderID):
    b_cell_match: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="B-Cell Match",
        description="What was the result of the b-cell match test?",
        isin=testres,
    )
    b_cell_match_e: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="B-Cell Match at e",
        description="What was the result of the b-cell match test? Marked with an e",
        isin=testres,
    )
    b_cell_match_h: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="B-Cell Match at h",
        description="What was the result of the b-cell match test? Marked with an h",
        isin=testres,
    )
    crossmatch_program: Series[str] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Crossmatch Program",
        description="In which program was this match calculated?",
        isin=[
            "Crossmatch (ETKAS) zur Allokation der Nieren",
            "Crossmatch (ETKAS) vor Transplantation einer Niere",
            "Crossmatch (ESP) zur Allokation der Nieren",
            "Crossmatch (ESP) vor Transplantation einer Niere",
        ],
    )
    date_e: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Date for e",
        description="When was the e result determined?",
    )
    date_h: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Date for h",
        description="When was the h result determined?",
    )
    date_s: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Date for s",
        description="When was the s result determined?",
    )
    esp_match: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="ESP Match",
        description="What was the result part of the ESP match test?",
        isin=["no", "yes"],
    )
    exam_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Schedule Date",
        description="When was this test ordered?",
    )
    igg_match: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="IgG Match",
        description="What was the result of the IgG match test?",
        isin=testres,
    )
    igg_match_e: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="IgG Match at e",
        description="What was the result of the IgG match test? Marked with an e",
        isin=testres,
    )
    igg_match_h: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="IgG Match at h",
        description="What was the result of the IgG match test? Marked with an h",
        isin=testres,
    )
    igm_match: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="IgM Match",
        description="What was the result of the IgM match test?",
        isin=testres + ["Ersatzempfänger"],
    )
    igm_match_h: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="IgM Match at h",
        description="What was the result of the IgM match test? Marked with an h",
        isin=testres,
    )
    match_dso_running_number: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="1. Running number",
        description="The number this compatibility test had in a set",
    )
    match_dso_running_number: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Rank",
        description="The rank this compatibility test had in a set",
    )
    match_running_id: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="2. Running number",
        description="The number this compatibility test had in a set",
    )
    result: Series[str] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Match Test Result",
        description="The result of this match test",
        isin=testres + ["Ersatzempfänger"],
    )
    result_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Result Date",
        description="When was this result determined?",
    )
    t_cell_match: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="T-Cell Match",
        description="What was the result of the T-cell match test?",
        isin=testres,
    )
    t_cell_match_e: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="B-Cell Match at e",
        description="What was the result of the T-cell match test? Marked with an e",
        isin=testres,
    )
    t_cell_match_h: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="T-Cell Match at h",
        description="What was the result of the T-cell match test? Marked with an h",
        isin=testres,
    )
    t_cell_match: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="T-Cell Match",
        description="What was the result of the T-cell match test?",
        isin=testres,
    )

    class Config:
        title = "Crossmatch Testing Dataset"
        description = "Each row represents a performed match test between potential donor and recipient pair. The data is based on the 'element_spender_postmortem_labor_crossmatch.csv' file. It contains data from the DSO."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemLabCrossmatch, data)

In [ ]:
DonorPostmortemLabCrossmatch.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemLabCrossmatch.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)